# Universal Data Acquisition Pipeline

This notebook serves as a universal wrapper to convert any geothermal dataset (.xlsx) into the standardized schema.

**Instructions:**
1. Place your raw `.xlsx` file in the `Input` folder.
2. Enter the filename below.
3. Define Column Mapping (which column corresponds to which standard attribute?).
4. Start Pipeline. The result will appear in the `Output` folder.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from pipeline_logic import GeothermalDataPipeline
import schema_config

## 1. Configuration

In [ ]:
# ----------------------- 1.1 Define Paths -----------------------
# Place your raw .xlsx file in the "Input" folder.
# The result will be saved in the "Output" folder.

INPUT_FILENAME = "My_Dataset.xlsx" 
OUTPUT_FILENAME = "My_Dataset_Clean.xlsx"

# ----------------------- Automatic path construction -----------------------
base_dir = Path.cwd()
INPUT_EXCEL_PATH = base_dir / "Input" / INPUT_FILENAME
OUTPUT_EXCEL_PATH = base_dir / "Output" / OUTPUT_FILENAME

print(f"Input Path:  {INPUT_EXCEL_PATH}")
print(f"Output Path: {OUTPUT_EXCEL_PATH}")

# ----------------------- Optional: Sheet name in the Excel file -----------------------
SHEET_NAME = 0 

In [ ]:
# ----------------------- 1.2 Define Column Mapping -----------------------
# Left: Column name in your input file (approximate, case-insensitive)
# Right: Name of the target attribute (see list below or schema_config.py)

# NOTE: If a unit needs conversion (e.g. mg/L to mmol/L),
# map to "..._in_mg/L". The pipeline converts this automatically!

COLUMN_MAPPING = {
    # --- Examples ---
    # "Latitude": "WGS84_latitude",
    # "Temp C": "temperature_in_c",
    # "Sodium (mg/L)": "Na_in_mg/L",  # autom. conversion to Na_in_mmol/L
    
    # --- Your Mapping here ---
    "Location Name": "location",
    
}

### Available Target Attributes (Excerpt)
- `location`, `well_or_spring_name`
- `WGS84_latitude`, `WGS84_longitude`
- `temperature_in_c`, `pH`, `electrical_conductivity_25c_in_uS/cm`
- `Na_in_mmol/L`, `K_in_mmol/L`, `Ca_in_mmol/L`, `Mg_in_mmol/L`
- `Cl_in_mmol/L`, `SO4_in_mmol/L`, `HCO3_in_mmol/L`
- ... (see `schema_config.ATTRIBUTES`)

## 2. Execute Pipeline

In [ ]:
pipeline = GeothermalDataPipeline(
    input_path=INPUT_EXCEL_PATH,
    output_path=OUTPUT_EXCEL_PATH,
    column_mapping=COLUMN_MAPPING
)

pipeline.load_data(sheet_name=SHEET_NAME)
pipeline.normalize_input_columns()

# ----------------------- Optional: View normalized input columns before mapping -----------------------
print("Available Input Columns (normalized):")
print(pipeline.df.columns.tolist())

In [ ]:
# ----------------------- Apply mapping and convert units -----------------------
pipeline.apply_mapping()
pipeline.convert_units()

# ----------------------- Validate schema (warn if something is missing) -----------------------
pipeline.validate_schema()

# ----------------------- Preview -----------------------
display(pipeline.df.head())

In [ ]:
# ----------------------- Save -----------------------
pipeline.save_data()